#### BART
- transFormer 기반의 모델
- Encoder, Decoder 모두 사용하는 모델
- 장문의 테스트에서 요약 데이터를 생성하는데 사용하는 모델
- Encoder : 입력이 되는 데이터를 BERT형식으로 문장을 이해(순방향, 역방향)
- Decoder : 출력이 되는 요약문은 GPT형식으로 문장을 생성
- 해당 모델에서는 Tokenizer는 Sentencepeiece를 이용
- 인풋 tokenizer와 아웃풋 tokenizer를 따로 사용

In [23]:
# !pip install evaluate rouge_score

In [24]:
import numpy as np
from datasets import Dataset, DatasetDict
# 문장 간의 검증 지표를 만들어주는 라이브러리
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

In [32]:
# 미리 학습된 모델 산정 (kobert) -> 완벽한 모델 x
model_name = 'gogamza/kobart-base-v2'

In [33]:
# 학습에서 사용할 원문 데이터, 요약 데이터
train_docs = [
    '정부의 중소기업 세제 해택과 R&D 세액 공제를 확대한다고 밝혔다',
    '해당 기업은 분기 실적에서 매출 성장을 기록했으며 신제품 출시를 예고했다'
]

train_sums = [
    '정부가 중소기업 지원을 확대한다',
    '기업이 실적 개선과 신제품 출시를 발표했다'
]
valid_docs = [
    '교육부가 디지털 교과서 도입을 추천한다고 발표했다'
]
valid_sums = [
    '교육부가 디지털 교과서 도입을 추진했다'
]

In [34]:
# transfomer 모델에서 list형태의 데이터를 사용x -> Dataset
raw_ds = DatasetDict(
    {
        'train' : Dataset.from_dict(
            {
                'document' : train_docs,
                'summary' : train_sums
            }
        ),
        'validation' : Dataset.from_dict(
            {
                'document' : valid_docs,
                'summary' : valid_sums
            }
        )
    }
)

In [35]:
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 1
    })
})

In [36]:
# 토크나이저, 모델을 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 입력 / 출력 문장의 최대 길이 설정
max_input_len = 512
max_target_len = 128

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abohv\.cache\huggingface\hub\models--gogamza--kobart-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You pa

In [ ]:
# 데이터 전처리 -> 토큰화
def tok_fn(batch):
    # batch -> 인자값 -> 묶음형 데이터셋
    # raw_ds의 데이터들의 묶음 -> raw_ds에서 인풋 데이터 -> document의 데이터
    # 입력 데이터의 토큰화 -> 인코딩
    inputs = tokenizer(
        batch['document'],
        max_length = max_input_len,
        padding = 'max_length',         # 고정길이 패딩 사용
        truncation = True               # 최대 길이보다 긴 경우 자른다.
    )
    # 출력 데이터의 토큰화 -> 인코딩
    # 아웃풋의 토큰나이저
    with tokenizer.as_target_tokenizer